# Real-World Compilation Pipelines

This notebook demonstrates production-grade compilation patterns using qb-compiler.
Each section builds on real scenarios: parameter sweeps, budget-constrained experiments,
cross-vendor compilation, caching, and experiment manifests.

Topics:
1. End-to-end pipeline: build, check, compile, estimate, verify
2. Multi-circuit experiment: parameter sweep compilation
3. Budget-constrained experiment: maximise circuits within a fixed budget
4. Cross-vendor compilation: same algorithm on IBM + Rigetti + IonQ
5. Production patterns: error handling, logging, caching
6. Experiment manifests and metadata
7. Calibration drift detection
8. Batch compilation and strategy comparison

In [ ]:
import json
import time
import logging
import datetime
from dataclasses import dataclass, asdict
from pathlib import Path

from qb_compiler import (
    QBCircuit,
    QBCompiler,
    CompilerConfig,
    BACKEND_CONFIGS,
    check_viability,
    BackendRecommender,
)
from qb_compiler.cost.estimator import CostEstimator
from qb_compiler.cost.budget_optimizer import BudgetOptimizer
from qb_compiler.exceptions import (
    QBCompilerError,
    InvalidCircuitError,
    BackendNotSupportedError,
    BudgetExceededError,
)
from qb_compiler.integrations.qubitboost import detect_circuit_type, recommend_gates

## 1. End-to-end pipeline

The canonical workflow: build a circuit, check viability, compile, estimate cost,
and verify the result.

In [ ]:
# Step 1: Build a circuit
circ = QBCircuit(4)
circ.h(0).cx(0, 1).cx(1, 2).cx(2, 3).measure_all()

print(f"Built: {circ}")
print(f"  Qubits: {circ.n_qubits}, Gates: {circ.gate_count}, Depth: {circ.depth}, 2Q: {circ.two_qubit_count}")

# Step 2: Check viability on target backend
from qiskit import QuantumCircuit

qiskit_circ = QuantumCircuit(4, name="ghz_4")
qiskit_circ.h(0)
for i in range(3):
    qiskit_circ.cx(i, i + 1)
qiskit_circ.measure_all()

viability = check_viability(qiskit_circ, backend="ibm_fez")
print(f"\nViability: {viability.status}")
print(f"  Estimated fidelity: {viability.estimated_fidelity:.4f}")
print(f"  Signal/noise: {viability.signal_to_noise:.1f}x")

In [ ]:
# Step 3: Compile
compiler = QBCompiler.from_backend("ibm_fez")
result = compiler.compile(circ)

print(f"Original depth: {result.original_depth} -> Compiled: {result.compiled_depth} ({result.depth_reduction_pct:.1f}% reduction)")
print(f"Estimated fidelity: {result.estimated_fidelity:.4f}")
print(f"Compilation time: {result.compilation_time_ms:.1f} ms")

if result.pass_log:
    print(f"\nPass log:")
    for p in result.pass_log:
        print(f"  {p.pass_name:20s}  depth {p.depth_before} -> {p.depth_after}  ({p.elapsed_ms:.1f} ms)")

# Step 4: Estimate cost
estimator = CostEstimator()
cost = estimator.estimate("ibm_fez", shots=4096)
print(f"\nCost: {cost.shots:,} shots x ${cost.cost_per_shot:.5f} = ${cost.total_cost_usd:.4f}")

# Step 5: Verify
assert result.estimated_fidelity > 0.01, "Fidelity too low"
assert result.compiled_depth <= result.original_depth, "Compilation increased depth"
print(f"\nVerification passed.")

## 2. Multi-circuit experiment: parameter sweep

Compile the same circuit structure with different parameter values. Typical for
QAOA (varying gamma/beta) or VQE (scanning energy landscape).

In [ ]:
def build_qaoa_circuit(n_qubits, gamma, beta):
    """Build a single-layer QAOA circuit with given parameters."""
    circ = QBCircuit(n_qubits)
    for q in range(n_qubits):
        circ.h(q)
    for q in range(n_qubits - 1):
        circ.cx(q, q + 1)
        circ.rz(q + 1, gamma)
        circ.cx(q, q + 1)
    for q in range(n_qubits):
        circ.rx(q, beta)
    circ.measure_all()
    return circ


gamma_values = [0.1 * i for i in range(1, 11)]
compiler = QBCompiler.from_backend("ibm_fez")
sweep_results = []

print(f"{'gamma':>6s} | {'orig_d':>6s} | {'comp_d':>6s} | {'fidelity':>8s} | {'time_ms':>8s}")
print("-" * 50)

for gamma in gamma_values:
    circ = build_qaoa_circuit(4, gamma, 0.5)
    result = compiler.compile(circ)
    sweep_results.append({
        "gamma": gamma, "beta": 0.5,
        "original_depth": result.original_depth,
        "compiled_depth": result.compiled_depth,
        "fidelity": result.estimated_fidelity,
        "time_ms": result.compilation_time_ms,
    })
    print(
        f"{gamma:>6.1f} | {result.original_depth:>6d} | {result.compiled_depth:>6d} | "
        f"{result.estimated_fidelity:>8.4f} | {result.compilation_time_ms:>8.1f}"
    )

print(f"\nCompiled {len(sweep_results)} circuits")

## 3. Budget-constrained experiment

Given a fixed budget (e.g. $50), determine how many circuits can be run
and on which backend. The `BudgetOptimizer` handles this.

In [ ]:
budget_usd = 50.0
target_shots = 4096
optimizer = BudgetOptimizer(min_shots=100)

print(f"Budget: ${budget_usd:.2f}, Target shots/circuit: {target_shots:,}\n")
print(f"{'Backend':>16s} | {'$/circuit':>10s} | {'Max circuits':>12s} | {'Strategy':>16s}")
print("-" * 65)

for name in sorted(BACKEND_CONFIGS.keys()):
    try:
        opt = optimizer.optimize(name, budget_usd=budget_usd, target_shots=target_shots, circuit_depth=20)
        cost_per_circuit = opt.estimated_cost_usd
        max_circuits = int(budget_usd / cost_per_circuit) if cost_per_circuit > 0 else 999
        print(f"{name:>16s} | ${cost_per_circuit:>9.4f} | {max_circuits:>12,d} | {opt.strategy:>16s}")
    except Exception:
        print(f"{name:>16s} | {'OVER BUDGET':>10s} | {'0':>12s} | —")

# Find the cheapest backend that meets requirements
print()
best = optimizer.find_cheapest_backend(budget_usd=budget_usd, min_qubits=4, target_shots=target_shots)

if best:
    circuits_affordable = int(budget_usd / best.estimated_cost_usd)
    print(f"Recommended: {best.backend}")
    print(f"  Can run {circuits_affordable} circuits at {best.recommended_shots:,} shots each")
    print(f"  Strategy: {best.strategy}")
    print(f"  Estimated fidelity: {best.estimated_fidelity:.3f}")
else:
    print("No backend fits within budget.")

## 4. Cross-vendor compilation

Compile the same algorithm for IBM, Rigetti, IonQ, and IQM to compare how
different gate sets and topologies affect the compiled circuit.

In [ ]:
circ = QBCircuit(4)
circ.h(0).cx(0, 1).cx(1, 2).cx(2, 3)
for q in range(4):
    circ.rz(q, 0.5 + 0.2 * q)
circ.measure_all()

cross_vendor = ["ibm_fez", "ibm_torino", "rigetti_ankaa", "ionq_aria", "iqm_garnet"]

print(f"Source: {circ.n_qubits}q, {circ.gate_count} gates, depth {circ.depth}\n")
print(f"{'Backend':>14s} | {'Basis':>25s} | {'Depth':>6s} | {'Gates':>6s} | {'Fidelity':>8s} | {'Time':>7s}")
print("-" * 80)

for backend in cross_vendor:
    comp = QBCompiler.from_backend(backend)
    r = comp.compile(circ)
    spec = BACKEND_CONFIGS[backend]
    basis = ", ".join(spec.basis_gates[:4]) + ("..." if len(spec.basis_gates) > 4 else "")
    print(
        f"{backend:>14s} | {basis:>25s} | {r.compiled_depth:>6d} | "
        f"{r.compiled_circuit.gate_count:>6d} | {r.estimated_fidelity:>8.4f} | {r.compilation_time_ms:>6.1f}ms"
    )

## 5. Production patterns: error handling, logging, and caching

qb-compiler uses a typed exception hierarchy for precise error handling.
The `qb_compiler` logger emits INFO-level messages during compilation.

In [ ]:
def safe_compile(circuit, backend, budget_usd=None):
    """Compile with comprehensive error handling."""
    try:
        compiler = QBCompiler.from_backend(backend)
        result = compiler.compile(circuit, budget_usd=budget_usd)
        return {"status": "success", "result": result}
    except InvalidCircuitError as e:
        return {"status": "invalid_circuit", "error": str(e)}
    except BackendNotSupportedError as e:
        return {"status": "bad_backend", "error": str(e)}
    except BudgetExceededError as e:
        return {"status": "over_budget", "estimated": e.estimated_usd, "budget": e.budget_usd}
    except QBCompilerError as e:
        return {"status": "compiler_error", "error": str(e)}


valid_circuit = QBCircuit(2)
valid_circuit.h(0).cx(0, 1).measure_all()

print("Valid circuit:", safe_compile(valid_circuit, "ibm_fez")["status"])
print("Bad backend:  ", safe_compile(valid_circuit, "fake_backend")["status"])
print("Over budget:  ", safe_compile(valid_circuit, "ionq_aria", budget_usd=0.001)["status"])

# Enable logging to see compiler internals
logging.basicConfig(level=logging.INFO)
qbc_logger = logging.getLogger("qb_compiler")
qbc_logger.setLevel(logging.INFO)

circ = QBCircuit(3)
circ.h(0).cx(0, 1).cx(1, 2).measure_all()
compiler = QBCompiler.from_backend("ibm_fez")
result = compiler.compile(circ)
print(f"\nLogged result: depth {result.compiled_depth}, fidelity {result.estimated_fidelity:.4f}")

qbc_logger.setLevel(logging.WARNING)  # reset

In [ ]:
class CompilationCache:
    """Cache compilation results by circuit structure (ignoring parameters)."""

    def __init__(self):
        self._cache = {}
        self.hits = 0
        self.misses = 0

    def _key(self, circuit, backend):
        structure = tuple((op.name, op.qubits) for op in circuit.ops)
        return (circuit.n_qubits, structure, backend)

    def get_or_compile(self, circuit, backend):
        key = self._key(circuit, backend)
        if key in self._cache:
            self.hits += 1
            return self._cache[key]
        self.misses += 1
        compiler = QBCompiler.from_backend(backend)
        result = compiler.compile(circuit)
        self._cache[key] = result
        return result


cache = CompilationCache()
t0 = time.perf_counter()
for gamma in [0.1 * i for i in range(20)]:
    circ = build_qaoa_circuit(4, gamma, 0.5)
    result = cache.get_or_compile(circ, "ibm_fez")
elapsed = (time.perf_counter() - t0) * 1000

print(f"20 compilations in {elapsed:.0f}ms")
print(f"Cache hits: {cache.hits}, misses: {cache.misses}")
print(f"Cache hit rate: {cache.hits / (cache.hits + cache.misses) * 100:.0f}%")

## 6. Research experiment harness with manifest generation

A complete harness: build circuits, compile, analyze, and save a JSON experiment
manifest for reproducibility.

In [ ]:
@dataclass
class ExperimentConfig:
    name: str
    backend: str
    shots: int = 4096
    strategy: str = "fidelity_optimal"
    budget_usd: float | None = None


@dataclass
class CircuitResult:
    circuit_id: str
    n_qubits: int
    original_depth: int
    compiled_depth: int
    estimated_fidelity: float
    compilation_time_ms: float
    estimated_cost_usd: float
    viable: bool
    parameters: dict


def run_experiment(config, circuits):
    """Compile all circuits and return results with total cost."""
    compiler = QBCompiler.from_backend(config.backend, strategy=config.strategy)
    cost_est = CostEstimator()
    results = []
    total_cost = 0.0

    for circuit_id, (circ, params) in circuits.items():
        cr = compiler.compile(circ)
        cost = cost_est.estimate(config.backend, config.shots)

        if config.budget_usd and total_cost + cost.total_cost_usd > config.budget_usd:
            print(f"  Budget exceeded at {circuit_id}. Stopping.")
            break

        total_cost += cost.total_cost_usd
        results.append(CircuitResult(
            circuit_id=circuit_id, n_qubits=circ.n_qubits,
            original_depth=cr.original_depth, compiled_depth=cr.compiled_depth,
            estimated_fidelity=cr.estimated_fidelity,
            compilation_time_ms=cr.compilation_time_ms,
            estimated_cost_usd=cost.total_cost_usd,
            viable=cr.estimated_fidelity > 0.01, parameters=params,
        ))

    return results, total_cost


# Build and run experiment
experiment_circuits = {}
for gamma in [0.2, 0.4, 0.6, 0.8, 1.0]:
    circ = build_qaoa_circuit(4, gamma, 0.5)
    experiment_circuits[f"qaoa_g{gamma:.1f}"] = (circ, {"gamma": gamma, "beta": 0.5})

config = ExperimentConfig(name="qaoa_gamma_sweep", backend="ibm_fez", budget_usd=10.0)
results, total_cost = run_experiment(config, experiment_circuits)

print(f"Experiment: {config.name} | Backend: {config.backend} | Cost: ${total_cost:.4f}\n")
for r in results:
    print(f"  {r.circuit_id:>15s}  depth={r.compiled_depth:>3d}  fid={r.estimated_fidelity:.4f}  ${r.estimated_cost_usd:.4f}")

# Generate experiment manifest
manifest = {
    "experiment": config.name,
    "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "config": {"backend": config.backend, "shots": config.shots, "strategy": config.strategy, "budget_usd": config.budget_usd},
    "summary": {
        "total_circuits": len(results),
        "total_cost_usd": round(total_cost, 4),
        "viable_circuits": sum(1 for r in results if r.viable),
        "avg_fidelity": round(sum(r.estimated_fidelity for r in results) / len(results), 4) if results else 0,
    },
    "circuits": [asdict(r) for r in results],
}

print(f"\nManifest ({len(manifest['circuits'])} circuits):")
print(json.dumps(manifest, indent=2))

## 7. Calibration drift detection

Quantum hardware calibration drifts over time. Key indicators to re-compile:
- Calibration data older than `calibration_max_age_hours` (default: 24h)
- Median CX error changed by more than 20%
- Previous layout used qubits that now have poor error rates

In [ ]:
def should_recompile(
    current_cx_error: float,
    original_cx_error: float,
    hours_since_compile: float,
    max_age_hours: float = 24.0,
    error_drift_threshold: float = 0.20,
) -> tuple[bool, str]:
    """Determine whether a circuit should be recompiled."""
    if hours_since_compile > max_age_hours:
        return True, f"Calibration is {hours_since_compile:.1f}h old (max: {max_age_hours}h)"

    if original_cx_error > 0:
        drift = abs(current_cx_error - original_cx_error) / original_cx_error
        if drift > error_drift_threshold:
            return True, f"CX error drifted {drift:.0%} ({original_cx_error:.4f} -> {current_cx_error:.4f})"

    return False, "Compilation is still current"


scenarios = [
    ("Fresh compile",       0.005, 0.005, 2.0),
    ("Slight drift",        0.006, 0.005, 12.0),
    ("Significant drift",   0.008, 0.005, 12.0),
    ("Stale calibration",   0.005, 0.005, 30.0),
]

for name, current, orig, hours in scenarios:
    recompile, reason = should_recompile(current, orig, hours)
    status = "RECOMPILE" if recompile else "OK"
    print(f"  {name:25s}  [{status:>9s}]  {reason}")

## 8. Batch compilation and strategy comparison

Measure compilation throughput across backends, then compare the three
compilation strategies on a deeper circuit.

| Strategy | Optimization level | Best for |
|----------|-------------------|----------|
| `fidelity_optimal` | 3 (aggressive) | Production runs, expensive backends |
| `depth_optimal` | 2 (standard) | Circuits near decoherence limits |
| `budget_optimal` | 1 (light) | Large sweeps, exploratory work |

In [ ]:
def benchmark_compilation(n_circuits, n_qubits, backend, strategy="fidelity_optimal"):
    """Benchmark compilation throughput."""
    compiler = QBCompiler.from_backend(backend, strategy=strategy)
    circuits = []
    for i in range(n_circuits):
        c = QBCircuit(n_qubits)
        for q in range(n_qubits):
            c.h(q)
        for q in range(n_qubits - 1):
            c.cx(q, q + 1)
        for q in range(n_qubits):
            c.rz(q, 0.1 * (i + 1))
        c.measure_all()
        circuits.append(c)

    t0 = time.perf_counter()
    results = [compiler.compile(c) for c in circuits]
    total_ms = (time.perf_counter() - t0) * 1000

    return {
        "backend": backend, "n_qubits": n_qubits, "strategy": strategy,
        "total_ms": round(total_ms, 1),
        "per_circuit_ms": round(total_ms / n_circuits, 1),
        "circuits_per_sec": round(n_circuits / (total_ms / 1000), 1),
    }


configs = [
    (50, 4, "ibm_fez", "fidelity_optimal"),
    (50, 4, "ibm_fez", "budget_optimal"),
    (50, 8, "ibm_fez", "fidelity_optimal"),
    (50, 4, "rigetti_ankaa", "fidelity_optimal"),
]

print(f"{'Backend':>14s} | {'Qubits':>6s} | {'Strategy':>16s} | {'Total':>8s} | {'Per circ':>8s} | {'circ/s':>7s}")
print("-" * 75)

for n, nq, backend, strategy in configs:
    b = benchmark_compilation(n, nq, backend, strategy)
    print(
        f"{b['backend']:>14s} | {b['n_qubits']:>6d} | {b['strategy']:>16s} | "
        f"{b['total_ms']:>7.0f}ms | {b['per_circuit_ms']:>7.1f}ms | {b['circuits_per_sec']:>6.0f}/s"
    )

In [ ]:
# Strategy comparison on a deeper circuit
circ = QBCircuit(6)
for q in range(6):
    circ.h(q)
for layer in range(3):
    for q in range(5):
        circ.cx(q, q + 1)
    for q in range(6):
        circ.rz(q, 0.3 * (layer + 1))
circ.measure_all()

print(f"Source: {circ.n_qubits}q, {circ.gate_count} gates, depth {circ.depth}\n")

for strategy in ["fidelity_optimal", "depth_optimal", "budget_optimal"]:
    compiler = QBCompiler.from_backend("ibm_fez", strategy=strategy)
    result = compiler.compile(circ)
    print(
        f"  {strategy:>18s}: depth {result.original_depth} -> {result.compiled_depth} "
        f"({result.depth_reduction_pct:+.1f}%)  "
        f"fidelity={result.estimated_fidelity:.4f}  "
        f"time={result.compilation_time_ms:.1f}ms"
    )

## Summary

### Pipeline checklist

1. **Build** your circuit using `QBCircuit` or Qiskit
2. **Check viability** with `check_viability()` before spending QPU time
3. **Select backend** using `BackendRecommender` or `BudgetOptimizer`
4. **Compile** with `QBCompiler.compile()` using the appropriate strategy
5. **Estimate cost** with `CostEstimator` and enforce budgets
6. **Verify** the result meets your fidelity and depth requirements
7. **Save** compilation metadata as a manifest for reproducibility
8. **Monitor** calibration drift and re-compile when needed